## Strategy evaluation

- compare alternate student loan repayment / investmet strategies
- each strategy is evaluated on the same stochastic salary and investment scenarios

In [1]:
from quant_slc_hedging.data_model import LoanModelInputs, SalaryModelInputs, InvestmentModelInputs, SalaryGrowthType, salary_growth_amounts
from quant_slc_hedging.salary import SalaryModel
from quant_slc_hedging.loan import LoanModel, LoanModelResult
from quant_slc_hedging.strategies.base_strategy import Strategy
from quant_slc_hedging.strategies.min_repayment import MinRepaymentStrategy
from quant_slc_hedging.strategies.fixed_pct_repayment import FixedPctRepaymentStrategy
from quant_slc_hedging.strategies.index_fund import IndexFundStrategy
from quant_slc_hedging.simulation import SimulationHandler
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd
from typing import List, Union
from dataclasses import dataclass

In [2]:
n_paths = 10_000
years_remaining = 25
observations = years_remaining * 12
seed = 1234
starting_salary = 35_000
initial_loan = 45_000
initial_investment_balance = 5_000
salary_growth = SalaryGrowthType(growth_type="Medium")
annual_investment_pct = 0.05
annual_investment_vol = 0.04
payoff_loan_with_investments = True

In [3]:
salary_rng = np.random.default_rng(seed)
investment_rng = np.random.default_rng(seed + 1) 

salary_config=SalaryModelInputs(
    starting_salary=starting_salary,
    salary_growth_dist=salary_growth
    )
loan_config = LoanModelInputs(
    initial_loan_balance=initial_loan,
    remaining_loan_term_months=12*years_remaining
)
investment_config = InvestmentModelInputs(
    initial_investment_balance=initial_investment_balance,
    annual_expected_return=annual_investment_pct,
    annual_vol=annual_investment_vol,
    payoff_loan_with_investments=payoff_loan_with_investments
)


In [4]:
strategies = {
    "Minimum Repayment": MinRepaymentStrategy(),
    "5% Overpayment": FixedPctRepaymentStrategy(
        fixed_excess_pct=0.05,
        repayment_threshold=loan_config.repayment_threshold 
    ),
    "5% Invested Index Fund": IndexFundStrategy(
        investment_config=investment_config,
        investment_pct=0.05,
        repayment_pct=0.0,
        repayment_threshold=loan_config.repayment_threshold,
        rng_gen=investment_rng,
        n_paths=n_paths,
        n_obs=observations
    ),
    "5% Invested Index Fund, 5% Overpayment": IndexFundStrategy(
        investment_config=investment_config,
        investment_pct=0.05,
        repayment_pct=0.05,
        repayment_threshold=loan_config.repayment_threshold,
        rng_gen=investment_rng,
        n_paths=n_paths,
        n_obs=observations
    )
}

In [5]:
results = {}

for name, strategy in strategies.items():
    sim = SimulationHandler(salary_config=salary_config, loan_config=loan_config, investment_config=investment_config, strategy=strategy, n_paths=n_paths, observations=observations, salary_rng=salary_rng)
    results[name] = sim.run_simulation()

Sim finished
Sim finished
Sim finished
Sim finished


In [8]:
def calc_metrics(strategy_name, sim_res):
    lb = sim_res.loan_result.loan_balance
    interest = sim_res.loan_result.interest_accrued
    base = sim_res.loan_result.base_repayment
    additional = sim_res.loan_result.additional_repayment
    ib = sim_res.investment_balance
    
    # Repayment metrics
    expected_end_loan_balance = lb[:, -1].mean()
    med_end_loan_balance = lb[:, -1].median()
    expected_end_inv_balance = ib[:, -1].mean()
    med_end_inv_balance = ib[:, -1].median()
    expected_term_wealth = (lb[:, -1] - ib[:, -1]).mean()
    loan_repaid = np.any(lb== 0, axis=1)
    probability_repaid = loan_repaid.mean()
    repaid = np.any(lb == 0, axis=1)
    repayment_month = np.full(n_paths, np.nan)
    repayment_month[repaid] = np.argmax(
        lb[repaid] == 0,
        axis=1,
    )
    repayment_year = repayment_month / 12
    total_base_payments = np.sum(base, axis=1).mean()
    total_add_payments = np.sum(additional, axis=1).mean()
    total_interest_payment = np.sum(interest, axis=1).mean()

    repayment_metrics = {
        "end_loan_balance": expected_end_loan_balance,
        "med_end_loan_balance": med_end_loan_balance,
        "end_inv_balance": expected_end_inv_balance,
        "med_end_inv_balance": med_end_inv_balance,
        "end_net_wealth": expected_term_wealth,
        "probaility_repaid": probability_repaid,
        "probaility_repaid": probability_repaid,
    }
    print(repayment_metrics)

calc_metrics("5% Invested Index Fund", results["5% Invested Index Fund"])

AttributeError: 'numpy.ndarray' object has no attribute 'median'

In [ ]:
np.median()